# 📌 PEFT (Parameter-Efficient Fine-Tuning)

![Topic](https://img.shields.io/badge/Topic-PEFT-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Advanced-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-August%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — **PEFT (Parameter-Efficient Fine-Tuning) adapts a large pretrained model to a new task by freezing almost all of its weights and training only a small number of new or reparameterized parameters.** It captures most of the benefit of full fine-tuning at a tiny fraction of the compute, memory, and storage cost.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install numpy` (runnable demo below) · `pip install transformers peft accelerate` (for real fine-tuning, illustrative snippet) |


---
## 1. Overview

**PEFT is a family of techniques for adapting a large pretrained model to a new task or domain without updating all of its parameters.** Instead, it freezes the base model and trains a small set of new or reparameterized weights, often well under 1% of the original parameter count, that sit on top of or alongside the frozen network.

**Why does this matter?** 
Full fine-tuning a modern LLM means storing and updating gradients and optimizer state for every one of its parameters. That puts full fine-tuning out of reach for most individuals and even many organizations, and it also risks **catastrophic forgetting**, where the model overwrites general knowledge learned during pretraining while adapting to the narrow new task.

**There are two broad families of PEFT methods:**
* **Additive methods:** Freeze the base model and insert small new trainable modules into the network, such as adapter layers or trainable "virtual token" embeddings (prefix/prompt tuning).
* **Reparameterization methods:** Freeze the base weights and represent the *update* to each weight matrix as a low-rank decomposition, most famously **LoRA**, so only the small low-rank matrices are ever trained.

---
## 2. How It Works

### 2.1 Start with a frozen, pretrained model
The model already knows a lot from pretraining. LoRA keeps every original weight matrix `W` locked exactly as it is: nothing about it is allowed to change, and no gradients are ever computed for it. Think of `W` as a huge, expensive-to-move textbook that stays on the shelf untouched.

### 2.2 Pick which weight matrices to adapt
Rather than touching the whole model, LoRA targets a handful of matrices, typically the query/key/value/output projections inside the attention blocks, since these are most responsible for task-specific behavior. Everything else in the model isn't even considered.

### 2.3 Decompose the update into two small matrices
If you *were* going to fine-tune `W` normally, you'd learn a full correction matrix `ΔW` the same size as `W`, and for a large model that's millions of numbers. LoRA's trick is to never build that full-size correction. Instead it approximates `ΔW` as the product of two much smaller matrices:

`ΔW = B @ A`

where `A` is `r × d_in` and `B` is `d_out × r`. Picture it as a narrow hourglass: `A` first squeezes the input down to just `r` numbers (the "bottleneck"), and `B` expands that small bottleneck back out to the full output size. Because data is forced through that narrow middle, `A` and `B` together need far fewer parameters than a full `d_out × d_in` matrix; that's the entire source of LoRA's efficiency. The rank `r` (often 4–64) is the width of that bottleneck: a bigger `r` lets the adapter represent more complex corrections, but costs more parameters.

### 2.4 Initialize carefully
Before training starts, we want the adapter to do *nothing*: the model should behave exactly like the untouched base model. LoRA achieves this with a simple trick: `A` gets small random values, but `B` starts as **all zeros**. Since anything multiplied by a zero matrix is zero, `B @ A = 0` at initialization, so the correction is a no-op on day one. Training can then adjust `A` and `B` gradually from that safe starting point.

### 2.5 Train only `A` and `B` on the target task
During fine-tuning, the model's forward pass uses the frozen weight plus the small correction, scaled by a factor `α/r`:

`y = (W + (α/r) · B·A) x`

Here `α` is a hyperparameter that controls how strongly the correction is allowed to influence the output. Backpropagation only ever updates `A` and `B`; `W` never moves, so the vast majority of the original model stays exactly as pretrained.

### 2.6 Merge or keep separate at inference
Once training is done, there are two ways to use the result:
- **Merge** `B·A` directly into `W` (`W_new = W + (α/r)·B·A`) so you get one ordinary weight matrix again, with zero extra computation at inference time.
- **Keep it separate**, so the same frozen base model can swap between many different small adapters, one per task, like plugging in different USB drives without changing the computer itself.

![PERFT.png](../assets/PEFT.png)

---
### Common PEFT Methods

| Method | How it adapts | Typical trainable params |
|--------|----------------|---------------------------|
| **LoRA** | Injects a low-rank update `B·A` into selected weight matrices | ~0.1 – 1% |
| **QLoRA** | Runs LoRA on top of a **4-bit quantized**, frozen base model | ~0.1 – 1% (plus large memory savings from quantization) |
| **Prefix Tuning** | Prepends trainable "virtual token" vectors to the keys/values of every attention layer | < 0.1% |
| **Prompt Tuning** | Prepends trainable embeddings only at the input layer | < 0.1% |
| **(IA)³** | Learns per-channel rescaling vectors that multiply activations elementwise | < 0.01% |
| **AdaLoRA** | Dynamically reallocates the rank budget across layers during training | ~0.1 – 1% |

> All of these ship as ready-to-use configs in [Hugging Face `peft`](https://huggingface.co/docs/peft/main/en/index) — swapping methods is usually a one-line config change.

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Massive memory savings** | Training <1% of parameters means far less gradient and optimizer-state memory, since Adam alone stores two extra values per trainable weight. |
| 🟢 | **Cheap to store and share** | A LoRA adapter for a 7B model is often just a few megabytes, versus tens of gigabytes for a full fine-tuned checkpoint. |
| 🟢 | **Mitigates catastrophic forgetting** | Freezing the base weights preserves the general knowledge learned during pretraining. |
| 🟢 | **Composable** | Many task-specific adapters can be trained once and swapped in and out of the same frozen base model. |
| 🟢 | **Stacks with quantization** | QLoRA fine-tunes a 4-bit quantized base model, making it possible to fine-tune a 65B-parameter model on a single consumer GPU. |
| 🔴 | **Slightly lower ceiling** | On some tasks, full fine-tuning still edges out PEFT methods in final accuracy. |
| 🔴 | **Hyperparameter sensitivity** | Rank, alpha, dropout, and which layers to target all meaningfully affect results and require tuning. |
| 🔴 | **Inference overhead if unmerged** | Keeping adapters separate (for swappability) adds a small extra matrix multiply at inference versus a merged model. |
| 🔴 | **Method sprawl** | LoRA, (IA)³, prefix tuning, and others each trade off differently — there is no single method that is best for every model and task. |

---
## 4. Code Example

> **Goal:** First, an illustrative (non-runnable) snippet showing how LoRA is applied to a real model with `peft`. Then a small runnable simulation of what LoRA is actually doing under the hood.

In [ ]:
# Illustrative only — requires `transformers`, `peft`, `accelerate`,
# and a downloaded base model, so it is not executed in this notebook.

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")

lora_config = LoraConfig(
    r=8,                                   # rank of the low-rank matrices
    lora_alpha=16,                         # scaling factor (alpha / r)
    target_modules=["q_proj", "v_proj"],   # which weight matrices to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# trainable params: 4,194,304 || all params: 8,034,000,000 || trainable%: 0.0522


In [1]:
import numpy as np
np.random.seed(0)

# --- A toy "linear layer" you might find inside a transformer's attention block ---
d_in, d_out = 4096, 4096          # e.g. a typical attention projection matrix
rank = 8                          # LoRA rank (r) -- deliberately small

# Full fine-tuning would update every entry of this frozen weight matrix W
W = np.random.randn(d_out, d_in) * 0.02
full_finetune_params = W.size

# LoRA freezes W and injects two small low-rank matrices, A and B
# The update is delta_W = B @ A, so the effective weight becomes W + (alpha/r) * (B @ A)
A = np.random.randn(rank, d_in) * 0.02   # trainable, small random init
B = np.zeros((d_out, rank))              # trainable, zero init
alpha = 16
scaling = alpha / rank

lora_params = A.size + B.size

x = np.random.randn(d_in)  # a fake input activation
delta_W = B @ A
y_full = W @ x
y_lora = (W + scaling * delta_W) @ x   # identical to y_full while B is still zero

print(f"Full fine-tuning trainable params: {full_finetune_params:,}")
print(f"LoRA trainable params (rank={rank}):   {lora_params:,}")
print(f"Reduction: {100 * (1 - lora_params / full_finetune_params):.2f}%")
print(f"Outputs identical at init (B=0)?       {np.allclose(y_full, y_lora)}")

Full fine-tuning trainable params: 16,777,216
LoRA trainable params (rank=8):   65,536
Reduction: 99.61%
Outputs identical at init (B=0)?       True


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **PEFT trades a small amount of ceiling for a massive drop in cost.** Freezing the base model and training <1% of parameters gets most of full fine-tuning's benefit for a fraction of the memory and storage.
- **LoRA's zero-init `B` matrix is the key trick.** It guarantees the adapted model behaves identically to the base model before training starts, then learns a low-rank correction from there.
- **PEFT and quantization compose.** QLoRA fine-tunes a 4-bit quantized frozen model, which is what makes fine-tuning huge models feasible on a single consumer GPU.
- **Adapters are portable and swappable.** Because the base model never changes, one frozen model can host many small, task-specific adapters instead of many full duplicated checkpoints.
- **There's no single best method.** LoRA, prefix tuning, prompt tuning, and (IA)³ all make different memory/accuracy tradeoffs — the [`peft` docs](https://huggingface.co/docs/peft/main/en/index) are the best place to compare them for a given use case.

</div>